# 🕵️ Dia 23 — Operadores de busca para mapear a pegada de uma organização

**Investigador Cyber 90 Dias** · Fase 2 · Rank Jutsu `C`

> Baseado no *Day 23* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-2-osint-digital-footprint/day-23.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

A Marta perguntou: *"O que mais existe por aí com o nosso nome que a gente não sabe?"* Rafa digita `site:serraazul.example -site:www.serraazul.example` e aparece `intranet-antiga.`. O buscador lembra o que a cooperativa esqueceu.

Neste notebook você vai:

1. **Montar consultas** com um construtor por formulário (você copia e cola; buscadores proíbem automação).
2. **Jogar o quiz do fluxograma**: registra, pare aí ou achado defensivo?
3. **Montar o mapa de calor** host × tipo, só com contagens observadas.
4. **Cruzar hosts** com o que o grafo dos Dias 20 e 21 já tinha.

🐍 Só biblioteca padrão do Python. Nada para instalar. Rode as células de cima para baixo (Shift+Enter). Tudo o que é do caso é **fictício** (domínios `.example`, IPs de documentação).

## 1. Construtor de consultas

Preencha o domínio e os hosts já conhecidos. A célula gera o próximo passo do laço `-site:` e as consultas por tipo de arquivo. **Ela não busca nada**: os termos de uso dos buscadores proíbem consultas automatizadas (tema do Dia 33). Copie, cole no Google e no Bing, percorra as páginas e anote.

In [ ]:
#@title Construtor de consultas { display-mode: "form" }
dominio = "serraazul.example"  #@param {type:"string"}
hosts_conhecidos = "www.serraazul.example, portal.serraazul.example"  #@param {type:"string"}
tipos = "pdf, docx, xlsx, pptx"  #@param {type:"string"}

hosts = [h.strip() for h in hosts_conhecidos.split(",") if h.strip()]   # lista limpa
exclusoes = " ".join(f"-site:{h}" for h in hosts)                       # descasca os já vistos
print("Próxima volta do laço (Passo 2):")
print(f"  site:{dominio} {exclusoes}\n")

print("Tipos de documento por host (Passo 3):")
for h in hosts:
    for t in [t.strip() for t in tipos.split(",")]:
        print(f"  site:{h} filetype:{t}")

print("\nMenções de terceiros (Passo 4):")
print(f'  "{dominio}" -site:{dominio}')
print(f'  site:github.com "{dominio}"')
print("\n⚠️ Exposição sensível (Passo 6) SÓ se o domínio for seu ou houver permissão escrita:")
print(f'  site:{dominio} intitle:"index of"')
print(f"  site:{dominio} (ext:env OR ext:log OR ext:sql OR ext:bak)")

## 2. 🎮 O fluxograma em ação

Para cada resultado: **1** = registra e conta (publicado de propósito) · **2** = pare aí (não abre, não baixa, não usa) · **3** = achado defensivo (domínio próprio ou com permissão escrita: remover e pedir retirada do índice).

Contexto: a Marta autorizou por escrito a checagem de exposição em `serraazul.example`. O golpe e o GitHub não são seus.

In [ ]:
QUIZ = [
    ("Relatório anual 2025 (PDF) em www.serraazul.example", "1",
     "Publicado de propósito. Conta no mapa de calor."),
    ("'Planilha cooperados - CPF e telefone' em intranet-antiga.serraazul.example", "3",
     "Domínio com permissão escrita: achado defensivo. Registre URL e título, NÃO abra, avise a Marta."),
    ("config.php no repositório público iscakit-mirror", "2",
     "Não é seu e não foi publicado de propósito. Pare aí: registre o repositório, não abra o arquivo."),
    ("Post num fórum público alertando sobre o SMS do Pix bloqueado", "1",
     "Menção de terceiro publicada de propósito. Registre e capture (Dia 19)."),
    ("'Index of /backup' em intranet-antiga.serraazul.example", "3",
     "Listagem de diretório exposta no domínio autorizado. Achado defensivo; não navegue pelos arquivos."),
    ("Formulário de 'regularização' em serraazul-pix.example", "1",
     "O golpe publicou para ser visto. É o próprio objeto do caso; já capturado no Dia 19."),
]
MODO_JOGO = True
acertos = 0
for item, certo, porque in QUIZ:
    print(f"\n• {item}")
    if MODO_JOGO:
        r = ""
        while r not in ("1", "2", "3"):
            r = input("   1 registra / 2 pare aí / 3 achado defensivo: ").strip()
        acertos += (r == certo)
        print("   ✅" if r == certo else "   ❌", end="")
    print(f"   {['registra', 'pare aí', 'achado defensivo'][int(certo) - 1]}: {porque}")
if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(QUIZ)}")

## 3. Mapa de calor host × tipo

Resultados sintéticos idênticos a [`caso-continuo/dados/dia-23/resultados-busca-serraazul.csv`](https://github.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/tree/main/caso-continuo/dados/dia-23). Se você tiver o seu próprio CSV com as mesmas colunas, troque `ARQUIVO`.

In [ ]:
import csv
from pathlib import Path
from urllib.parse import urlparse
from collections import defaultdict

def resultados():
    """Resultados sintéticos de busca (determinísticos). Tudo fictício."""
    R = []
    def add(consulta, buscador, url, titulo, tipo):
        R.append((consulta, buscador, url, titulo, tipo))
    # (host, tipo, quantidade observada, buscadores que acharam)
    plano = [
        ("www.serraazul.example", "pdf", 50, ("Google", "Bing")),   # 50 = teto de 5 páginas
        ("www.serraazul.example", "docx", 6, ("Google", "Bing")),
        ("www.serraazul.example", "xlsx", 2, ("Google",)),
        ("www.serraazul.example", "pptx", 3, ("Google", "Bing")),
        ("portal.serraazul.example", "pdf", 4, ("Google", "Bing")),
        ("eventos.serraazul.example", "pdf", 3, ("Bing",)),
        ("eventos.serraazul.example", "pptx", 2, ("Bing",)),
        ("intranet-antiga.serraazul.example", "pdf", 1, ("Google",)),
        ("intranet-antiga.serraazul.example", "docx", 9, ("Google",)),
        ("intranet-antiga.serraazul.example", "xlsx", 4, ("Google",)),
    ]
    titulos = {"pdf": "Relatório anual {i}", "docx": "Ata de assembleia {i}",
               "xlsx": "Tabela de tarifas {i}", "pptx": "Apresentação institucional {i}"}
    for host, tipo, qtd, buscadores in plano:
        for b in buscadores:
            for i in range(1, qtd + 1):
                add(f"site:{host} filetype:{tipo}", b, f"https://{host}/docs/{tipo}-{i:02d}.{tipo}",
                    titulos[tipo].format(i=i), tipo)
    # O achado sensível: planilha de cooperados indexada (NÃO ABRIR)
    add("site:intranet-antiga.serraazul.example filetype:xlsx", "Google",
        "https://intranet-antiga.serraazul.example/rh/cooperados-2025.xlsx",
        "Planilha cooperados - CPF e telefone", "xlsx")
    add('site:serraazul.example intitle:"index of"', "Google",
        "https://intranet-antiga.serraazul.example/backup/", "Index of /backup", "html")
    # Menções de terceiros (lado do golpe)
    add('"serraazul-pix.example" -site:serraazul-pix.example', "Google",
        "https://forum-golpes.example/t/sms-serra-azul-pix-bloqueado", "SMS falso Serra Azul - Pix bloqueado (alerta)", "html")
    add('"serraazul-pix.example" -site:serraazul-pix.example', "Bing",
        "https://noticias-regionais.example/2026/10/golpe-pix-cooperativa", "Cooperativa alerta para golpe do Pix bloqueado", "html")
    add('"sua chave Pix foi bloqueada" "Serra Azul"', "Google",
        "https://forum-golpes.example/t/sms-serra-azul-pix-bloqueado", "SMS falso Serra Azul - Pix bloqueado (alerta)", "html")
    add('site:github.com "IscaKit"', "Google",
        "https://github.example/usuario-anonimo/iscakit-mirror", "iscakit-mirror: kit de phishing (espelho)", "html")
    add('site:github.com "IscaKit"', "Google",
        "https://github.example/usuario-anonimo/iscakit-mirror/blob/main/config.php", "iscakit-mirror/config.php", "php")
    return R

ARQUIVO = "resultados-busca-serraazul.csv"       # troque pelo seu, se tiver
if not Path(ARQUIVO).exists():
    with open(ARQUIVO, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["consulta", "buscador", "url", "titulo", "tipo"])
        w.writerows(resultados())
with open(ARQUIVO, newline="", encoding="utf-8") as f:
    RES = list(csv.DictReader(f))
print(len(RES), "resultados lidos")

DOMINIO = "serraazul.example"
TIPOS = ["pdf", "docx", "xlsx", "pptx"]
NAO_ABRIR = ("cooperados", "config.php", "index of")      # sinais de material não intencional

urls_por_celula = defaultdict(set)      # (host, tipo) → URLs DISTINTAS vistas (Google e Bing somados)
buscadores_do_host = defaultdict(set)   # host → {"Google", "Bing"}
nao_abertos = []
for r in RES:
    host = urlparse(r["url"]).hostname
    if not host.endswith(DOMINIO):
        continue                          # menções de terceiros ficam para a seção 4
    buscadores_do_host[host].add(r["buscador"])
    if any(s in (r["titulo"] + r["url"]).lower() for s in NAO_ABRIR):
        nao_abertos.append(r)             # vai para o registro como URL + título, sem conteúdo
    if r["tipo"] in TIPOS:
        urls_por_celula[(host, r["tipo"])].add(r["url"])   # set: a mesma URL nos 2 buscadores conta 1 vez

hosts = sorted(buscadores_do_host)
TETO = 50                                 # o original: no máximo 5 páginas percorridas
def celula(h, t):
    return min(len(urls_por_celula[(h, t)]), TETO)

blocos = " ░▒▓█"                          # escala de cor em texto
print(f"\n{'Host':<36}" + "".join(f"{t:>8}" for t in TIPOS))
for h in hosts:
    linha = f"{h:<36}"
    for t in TIPOS:
        n = celula(h, t)
        cor = blocos[min(4, (n * 4 + TETO - 1) // TETO)] if n else " "
        linha += f"{cor * 2}{n:>6}"
    print(linha)
print("\nLegenda: contagens OBSERVADAS (URLs distintas percorridas, teto de 50), não estimativas 'Aproximadamente N'.")

In [ ]:
# Salva o CSV do mapa de calor para colar na planilha (Passo 5)
with open("mapa-de-calor-serraazul.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["host"] + TIPOS)
    for h in hosts:
        w.writerow([h] + [celula(h, t) for t in TIPOS])
print("💾 mapa-de-calor-serraazul.csv (aplique a escala de cores na planilha)\n")

# Registro dos achados sensíveis: URL e título, NADA de conteúdo
from datetime import datetime, timezone
agora = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
print("Achados defensivos (não abertos):")
for r in nao_abertos:
    print(f"  {agora} | {r['url']} | título no buscador: '{r['titulo']}' | NÃO ABERTO → avisar Marta")

## 4. Quem achou o quê, e o que o grafo não tinha

In [ ]:
GRAFO_D20_D21 = {"serraazul-pix.example", "www.serraazul-pix.example", "admin.serraazul-pix.example",
                 "cdn.serraazul-pix.example", "serraazul-seguranca.example", "coopserra-azul.example"}
print(f"{'Host':<36}{'Achado por':<16}{'No grafo?'}")
for h in hosts:
    b = buscadores_do_host[h]
    quem = "os dois" if len(b) == 2 else next(iter(b))
    print(f"{h:<36}{quem:<16}{'sim' if h in GRAFO_D20_D21 else 'NÃO → acrescentar ao registro'}")

print("\nMenções de terceiros:")
for r in RES:
    host = urlparse(r["url"]).hostname
    if not host.endswith(DOMINIO):
        aviso = "  ⛔ pare aí: não abrir" if "config.php" in r["url"] else ""
        print(f"  [{r['buscador']}] {r['url']}{aviso}")

print("\n✍️ Passo 7: escreva a frase sobre eventos.serraazul.example (só o Bing achou). Hipóteses: robots.txt, "
      "indexação recente, o Google tratou como duplicado.")

### 🧠 Camada 8

O título "Planilha cooperados - CPF e telefone" cria uma **lacuna de informação** (Loewenstein): a vontade de abrir é quase física, e vem disfarçada de dever ("preciso saber quantos"). Você não precisa. Decida antes da tentação.

## ✅ Antes de fechar o Dia 23

- [ ] Consultas montadas e rodadas à mão no Google **e** no Bing
- [ ] Mapa de calor com legenda "contagens observadas"
- [ ] Lista de hosts com o buscador que achou cada um
- [ ] Planilha de cooperados e `config.php` registrados como **não abertos**
- [ ] Frase do Passo 7 escrita

**Amanhã (Dia 24):** enumeração de username. Em você mesmo, e só em você. Rafa vai pedir a precisão de cada ferramenta, medida na sua própria pegada.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>